## 02_model_development

## Reading this notebook

Notebook 01 prepared the data. This notebook builds the models — and, more
importantly, produces the evidence that the models are worth trusting.

**What this notebook is actually for:**

It is tempting to read this as "train four classifiers and pick the best one."
That is not the point. The point is to answer three governance questions:

1. **Does the model actually predict what it claims to predict?** — after removing
   features that would not exist at a real credit decision (leakage), what is the
   honest performance?
2. **Is the model's confidence trustworthy?** — a model can rank applicants well
   (good AUC) and still produce probabilities that are systematically wrong. That
   distinction matters enormously for pricing and provisioning.
3. **Does the model treat groups differently?** — and if so, how badly, and what
   does that mean for deployment?

**How the notebook is structured:**

The phases follow the lifecycle of a model development exercise, but each phase is
also a governance artifact:

- Phases 0–3: setup, caveat, and loading the feature inventory from Notebook 01
- Phase 4: removing leakage features and cleaning the data
- Phases 5–9: encoding, splitting, and handling the class imbalance
- Phase 10: training four candidate models
- **Phase 11: quantifying the cost of leakage** — the headline finding
- Phase 12: honest performance evaluation
- Phase 13: calibration assessment
- **Phase 14: fairness testing** — the finding that blocks deployment
- Phases 15–18: saving artifacts and consolidating findings

**Two phases to read closely:**

Phase 11 (leakage impact) and Phase 14 (fairness). Phase 11 shows what happens
when you stop letting the model cheat. Phase 14 shows why a model can be accurate
on average and still be unusable.

**A note on version 2.0:**

This notebook is labelled V2.0 because it responds to findings raised in an
earlier round of review. The V2.0 changes are listed in Phase 0. Some of them are
fully implemented (leakage removal, fairness testing); some are documented but
not yet complete (the time-based split). The findings register in Phase 18 is
honest about which is which — and that honesty is itself part of the governance
story.

### Why this phase exists

A model development notebook that begins by training models has already skipped
the most important step. This phase states, before any code runs, what the
notebook is trying to accomplish and what regulatory frameworks it is being held
to.

The V2.0 change list is doing specific work: it records what was wrong in the
previous version and what has been changed in response. That is a governance
pattern — version control with documented rationale, not just version numbers.
A reviewer should be able to see not only that the notebook changed, but *why*.

**The honest caveat:** Not every V2.0 item is fully closed. The executive summary
lists "time-based split" as a V2.0 upgrade, but the notebook falls back to a
random split because no usable date column survives the cleaning in Notebook 01.
That gap is documented in Phase 7 and Phase 16. Calling it out here — rather than
letting a reviewer discover it — is the difference between a notebook that claims
compliance and one that demonstrates it.

### Why this phase exists

Same principle as Notebook 01: a model that cannot be reproduced cannot be
validated. The random seed is fixed once, at the top, and referenced everywhere.

**One thing to notice:** the seed governs the train/test split, the cross-validation
folds, and the initialization of all four model types. If any of those used a
different seed, a reviewer rerunning the notebook would get different metrics —
and would have no way to tell whether the difference came from the seed or from
something substantive.

## Phase 1 - Environment & Reproducibility

In [1]:
print("\n" + "=" * 80)
print("PHASE 1: ENVIRONMENT SETUP & REPRODUCIBILITY")
print("=" * 80)

"""
DECISION POINT 0: RANDOM SEED DOCUMENTATION
============================================
Decision: Project-wide random seed = 42

Rationale:
1. Ensures reproducibility across all stochastic operations
2. Consistent with Notebook 01
3. Enables stakeholders to reproduce our analysis

Regulatory Reference:
- OSFI B-13 s.3.2: Reproducibility requirement

Applied To:
- Train/test split (time-based with random tie-breaking)
- Cross-validation folds
- XGBoost, Random Forest, LightGBM
- Any stochastic operations
"""

RANDOM_STATE = 42
print(f"[OK] Project-wide random seed: {RANDOM_STATE}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
import json
import codecs
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

# ML libraries
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    average_precision_score, brier_score_loss, log_loss
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer

import xgboost as xgb
import lightgbm as lgb

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

# Create directories
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/reports", exist_ok=True)

print("[OK] Phase 1 Complete")


PHASE 1: ENVIRONMENT SETUP & REPRODUCIBILITY
[OK] Project-wide random seed: 42
[OK] Phase 1 Complete


#### What It Means

This sets up the Python environment, imports libraries, and documents the random seed for reproducibility.

#### Explain the Decision
- **Random seed = 42** Ensures all stochastic operations (train/test split, cross-validation, model training) produce the same results every time. This is a regulatory requirement (OSFI B-13 s.3.2).
- **Applied to all models** XGBoost, Random Forest, LightGBM all use random_state=42. This ensures consistency across models.

**I think about reproducibility before I write a single line of model code.**

## PHASE 2: DATASET CAVEAT (REFERENCED)

### Why this phase exists

Notebook 01 established the caveat; this phase re-references it rather than
restating it. That is deliberate: the caveat is a single source of truth, and
duplicating it across notebooks would eventually produce two versions that
disagree.

**The governance principle:** constraints that apply to a whole project should
live in one place and be referenced everywhere else. A caveat repeated in five
notebooks is a caveat that will drift.

In [2]:
print("\n" + "=" * 80)
print("PHASE 2: DATASET CAVEAT (REFERENCED)")
print("=" * 80)

"""
===============================================================================
DATASET CAVEAT (REFERENCED)
===============================================================================

This notebook uses the dataset described in Notebook 01. As stated there,
this project uses public Lending Club data as a PROXY for proprietary bank data.

For the full caveat, see Notebook 01 — Section 2: Dataset Caveat.

US/CANADA APPLICABILITY GAP:
- Lending Club is U.S. consumer lending data
- Canadian regulatory context (OSFI) is applied as a FRAMEWORK reference
- Model requires revalidation for Canadian portfolios

===============================================================================
"""

print("[OK] Dataset Caveat referenced from Notebook 01")


PHASE 2: DATASET CAVEAT (REFERENCED)
[OK] Dataset Caveat referenced from Notebook 01


#### What It Means

This references the dataset caveat from Notebook 1 — Lending Club data is a proxy for real bank data, not the real thing.

#### Explain the Decision
- **Referenced, not repeated** Saves space while maintaining honesty about data limitations.
- **US/Canada gap acknowledged** Shows you understand this is U.S. data being analyzed under a Canadian regulatory framework.

**I maintain consistency across notebooks and don't hide data limitations."

### PHASE 3: LOAD DATA & FEATURE INVENTORY

### Why this phase exists

This phase is where Notebook 01 and Notebook 02 connect. Rather than re-deriving
which features to exclude, this notebook *loads the feature inventory from
Notebook 01* — the exclusion decisions, the leakage flags, the reasons.

**Why this matters for governance:** The decision to exclude a feature should be
made once, documented once, and inherited everywhere downstream. If Notebook 02
re-decided which features to drop, there would be two records of the same
decision, and eventually they would differ. Loading the inventory enforces a
single audit trail.

**What to notice in the output:** 45 features excluded, 9 of them for leakage.
Those 9 are the ones that will be removed in Phase 4.

In [3]:
print("\n" + "=" * 80)
print("PHASE 3: LOADING DATA AND FEATURE INVENTORY")
print("=" * 80)

# Load the data
data_path = Path("data/loans_cleaned.csv")

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"[OK] Loaded {len(df):,} rows, {len(df.columns)} columns")
    print(f"  Default Rate: {df['default'].mean():.2%}")
else:
    # Fallback: try to load from original location
    data_path = Path("loans_full_schema.csv")
    if data_path.exists():
        df = pd.read_csv(data_path)
        print(f"[OK] Loaded {len(df):,} rows, {len(df.columns)} columns")
        print("  [WARNING] Using raw data - Notebook 01 should be run first")
    else:
        raise FileNotFoundError("data/loans_cleaned.csv not found. Please run Notebook 01 first.")

# Load feature inventory from Notebook 01
try:
    feature_inventory = pd.read_csv("data/feature_inventory.csv")
    print(f"[OK] Loaded feature inventory: {len(feature_inventory)} features")
    
    # Get excluded features
    excluded_features = feature_inventory[feature_inventory['Inclusion_Decision'] == 'EXCLUDE']['Feature'].tolist()
    leakage_features = feature_inventory[feature_inventory['Leakage_Flag'] == 'YES']['Feature'].tolist()
    
    print(f"  Excluded Features: {len(excluded_features)}")
    print(f"  Leakage Features: {len(leakage_features)}")
except FileNotFoundError:
    print("[WARNING] Feature inventory not found. Using default exclusion list.")
    excluded_features = ['Unnamed: 0', 'emp_title']
    leakage_features = ['balance', 'paid_total', 'paid_principal', 'paid_interest', 
                        'paid_late_fees', 'months_since_last_delinq', 'months_since_90d_late',
                        'months_since_last_credit_inquiry', 'issue_month']

# Load leakage findings from Notebook 01
try:
    leakage_findings = pd.read_csv("outputs/reports/leakage_findings.csv")
    print(f"[OK] Loaded leakage findings: {len(leakage_findings)} findings")
except FileNotFoundError:
    print("[WARNING] Leakage findings not found. Proceeding with default leakage list.")
    leakage_findings = pd.DataFrame()

print("\n[OK] Data and feature inventory loaded")


PHASE 3: LOADING DATA AND FEATURE INVENTORY
[OK] Loaded 10,000 rows, 57 columns
  Default Rate: 1.78%
[OK] Loaded feature inventory: 55 features
  Excluded Features: 45
  Leakage Features: 9
[OK] Loaded leakage findings: 9 findings

[OK] Data and feature inventory loaded


#### What It Means

This loads the cleaned data and feature inventory from Notebook 1. It shows:

- 10,000 rows of data
- 57 columns originally
- 1.78% default rate
- 45 features excluded (leakage + identifiers)
- 9 leakage features confirmed for removal

#### Explain the Decision
- **Load feature inventory** Creates traceability — you're using the same feature definitions as Notebook 1.
- **Confirm 9 leakage features** These are features that would NOT be available at origination. Removing them prevents look-ahead bias.

**I maintain consistency across notebooks. I don't redefine features — I load them from a single source of truth.**

### PHASE 4: DATA CLEANING & FEATURE ENGINEERING

### Why this phase exists

Three things happen here, and each is a governance decision, not just a
data-cleaning step.

**1. Leakage features are removed.** This is the phase where Notebook 01's
finding becomes an action. Nine features are dropped, each one printed by name,
so the removal is visible in the output rather than buried in a list
comprehension.

**2. Missing values are imputed.** Median for numeric, mode for categorical —
the same transparent strategy documented in Notebook 01. The output reports how
many values were imputed and with what, so the imputation is auditable.

**3. Outliers are capped at the 99th percentile.** This is worth pausing on,
because it is a modeling choice with fairness implications. Capping at the 99th
percentile compresses the extreme values of 29 features. If those extremes are
distributed differently across groups, capping can change the fairness picture.
The decision is defensible — it reduces the influence of outliers — but it should
be documented as a choice, not presented as neutral preprocessing.

**Feature engineering** adds four derived features: `dti_high`, `int_rate_high`,
`loan_to_income`, and `emp_years`. The first two are binary flags at
regulatory-relevant thresholds (DTI > 36%, interest rate > 15%). That is a
governance-relevant choice: encoding a threshold as a feature tells the model
"this boundary matters," which is a claim about the business, not just a
transformation.

**A quiet problem in the output:** `emp_years` is created from `emp_length`, but
by the time the data reaches Phase 8, `emp_years` is entirely NaN and gets
dropped. The feature engineering step produced a column that did not survive.
That is not fatal, but it is the kind of thing a reviewer will ask about — better
to note it than to let it be discovered.

In [4]:
print("\n" + "=" * 80)
print("PHASE 4: DATA CLEANING & FEATURE ENGINEERING")
print("=" * 80)

"""
DECISION POINT 1: LEAKAGE FEATURE REMOVAL
==========================================
Decision: Remove ALL leakage features from model training

Rationale:
1. Leakage features are NOT available at origination
2. Using them would cause look-ahead bias
3. Performance with leakage is INFLATED (0.865 AUC)
4. Performance without leakage is REALISTIC (0.621 AUC)

Regulatory Reference:
- OSFI E-23 Section 3.3: Data Quality
- SR 11-7: Model validation must identify data quality issues
"""

# ============================================================================
# 4.1 REMOVING DATA LEAKAGE FEATURES
# ============================================================================

print("\n4.1 REMOVING DATA LEAKAGE FEATURES:")
print("-" * 40)

# V2.0: Track removed features for documentation
leakage_features_removed = []

# Remove leakage features
for col in leakage_features:
    if col in df.columns:
        df = df.drop(columns=[col])
        leakage_features_removed.append(col)
        print(f"  - Removed: {col}")

if leakage_features_removed:
    print(f"\n[CRITICAL] Removed {len(leakage_features_removed)} leakage features:")
    print(f"  {leakage_features_removed}")

# Remove identifiers
drop_cols = ['Unnamed: 0', 'emp_title']
for col in drop_cols:
    if col in df.columns:
        df = df.drop(columns=[col])
        print(f"  - Removed identifier: {col}")

print(f"\n[SUMMARY] Total leakage features removed: {len(leakage_features_removed)}")

# ============================================================================
# 4.2 ROBUST MISSING VALUE HANDLING
# ============================================================================

print("\n4.2 HANDLING MISSING VALUES (ROBUST):")
print("-" * 40)

# Count missing values before
missing_before = df.isnull().sum().sum()
print(f"Missing values before imputation: {missing_before:,}")

# Separate numeric and categorical columns for imputation
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Remove target and identifiers from imputation lists
if 'default' in numeric_cols:
    numeric_cols.remove('default')
if 'loan_status' in categorical_cols:
    categorical_cols.remove('loan_status')

# Impute numeric columns with median
for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  Imputed {col} with median: {median_val:.2f}")

# Impute categorical columns with mode
for col in categorical_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0] if not df[col].mode().empty else 'Unknown'
        df[col].fillna(mode_val, inplace=True)
        print(f"  Imputed {col} with mode: {mode_val}")

# Verify no missing values remain
missing_after = df.isnull().sum().sum()
print(f"Missing values after imputation: {missing_after:,}")

if missing_after > 0:
    print("\n[WARNING] Some missing values remain. Dropping rows with missing values...")
    df = df.dropna()
    print(f"  Shape after dropping missing rows: {df.shape}")

# Double-check all columns
missing_cols = df.columns[df.isnull().any()].tolist()
if missing_cols:
    print(f"[ERROR] Columns still with missing values: {missing_cols}")
    raise ValueError("Missing values remain in dataset. Please check imputation.")
else:
    print("[OK] All missing values handled.")

# ============================================================================
# 4.3 Remove duplicates
# ============================================================================

print("\n4.3 REMOVING DUPLICATES:")
print("-" * 40)

duplicates_before = df.duplicated().sum()
if duplicates_before > 0:
    df = df.drop_duplicates()
    print(f"Removed {duplicates_before:,} duplicate rows")
else:
    print("No duplicates found")

# ============================================================================
# 4.4 Outlier treatment
# ============================================================================

print("\n4.4 OUTLIER TREATMENT:")
print("-" * 40)

outlier_capped = []
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    if col not in ['default']:
        q99 = df[col].quantile(0.99)
        original_max = df[col].max()
        df[col] = df[col].clip(upper=q99)
        if original_max > q99:
            outlier_capped.append(col)

if outlier_capped:
    print(f"Capped outliers in {len(outlier_capped)} features at 99th percentile")

# ============================================================================
# 4.5 Feature Engineering
# ============================================================================

print("\n4.5 FEATURE ENGINEERING:")
print("-" * 40)

if 'debt_to_income' in df.columns:
    df['dti_high'] = (df['debt_to_income'] > 36).astype(int)
    print("Created dti_high (DTI > 36%) - Regulatory affordability threshold")

if 'interest_rate' in df.columns:
    df['int_rate_high'] = (df['interest_rate'] > 15).astype(int)
    print("Created int_rate_high (Interest rate > 15%) - High-risk indicator")

if 'loan_amount' in df.columns and 'annual_income' in df.columns:
    df['loan_to_income'] = df['loan_amount'] / df['annual_income']
    df['loan_to_income'] = df['loan_to_income'].clip(upper=5)
    print("Created loan_to_income (Loan amount / Annual income) - Key affordability metric")

if 'emp_length' in df.columns:
    def convert_emp_length(x):
        if pd.isna(x):
            return np.nan
        if x == '10+ years':
            return 10
        if x == '< 1 year':
            return 0.5
        try:
            return float(x.split()[0])
        except:
            return np.nan
    
    df['emp_years'] = df['emp_length'].apply(convert_emp_length)
    df = df.drop(columns=['emp_length'])
    print("Converted emp_length to emp_years (numeric)")

print("\n[OK] Data cleaning and feature engineering complete")


PHASE 4: DATA CLEANING & FEATURE ENGINEERING

4.1 REMOVING DATA LEAKAGE FEATURES:
----------------------------------------
  - Removed: months_since_last_delinq
  - Removed: months_since_90d_late
  - Removed: months_since_last_credit_inquiry
  - Removed: issue_month
  - Removed: balance
  - Removed: paid_total
  - Removed: paid_principal
  - Removed: paid_interest
  - Removed: paid_late_fees

[CRITICAL] Removed 9 leakage features:
  ['months_since_last_delinq', 'months_since_90d_late', 'months_since_last_credit_inquiry', 'issue_month', 'balance', 'paid_total', 'paid_principal', 'paid_interest', 'paid_late_fees']
  - Removed identifier: Unnamed: 0
  - Removed identifier: emp_title

[SUMMARY] Total leakage features removed: 9

4.2 HANDLING MISSING VALUES (ROBUST):
----------------------------------------
Missing values before imputation: 26,714
  Imputed emp_length with median: 6.00
  Imputed debt_to_income with median: 17.57
  Imputed annual_income_joint with median: 113000.00
  Impute

#### What It Means

This section:

- Removes 9 leakage features (critical step)
- Handles missing values with median/mode imputation
- Removes duplicates (none found)
- Caps outliers at the 99th percentile
- Creates new features like dti_high, int_rate_high, loan_to_income

#### Explain the Decision
- **Remove 9 leakage features** This is the most critical step. Features like balance, paid_total, months_since_last_delinq are not available at origination.
- **Median imputation** Median is robust to outliers — unlike mean, it's not skewed by extreme values.
- **Mode imputation for categorical** Preserves the category distribution.
- **Capping at 99th percentile** Handles outliers without losing data (winsorization instead of deletion).
- **Create dti_high (DTI > 36%)** 36% is the Qualified Mortgage (QM) standard.
- **Create int_rate_high (>15%)** Interest rates above 15% are typically subprime.
- **Create loan_to_income** Key affordability metric — higher ratio = more stretched borrower.
- **Convert emp_length to numeric** Makes employment length usable in models.

**I understand regulatory thresholds and document my feature engineering decisions.**

### PHASE 5: ENCODE CATEGORICAL VARIABLES

### Why this phase exists

Machine learning models require numeric input, so categorical variables —
`state`, `homeownership`, `loan_purpose`, and others — are one-hot encoded.

**What one-hot encoding does:** it turns a categorical column into a set of
binary columns, one per category. `state` becomes `state_AL`, `state_AR`,
`state_AZ`, and so on. The model can then use each as a separate signal.

**The governance consequence, which is easy to miss:** one-hot encoding a
geographic variable means the model now has access to *state-level* information
as a direct input. That is exactly what makes the fairness testing in Phase 14
possible — and it is also what produces the severe disparate impact that phase
finds.

**In other words:** the encoding decision in this phase creates both the
capability to test for geographic fairness and the exposure that test detects.
This is not a coincidence; it is the mechanism. A model that uses state as a
feature will learn state-level patterns, including any that correlate with
protected characteristics.

In [5]:
print("\n" + "=" * 80)
print("PHASE 5: ENCODING CATEGORICAL VARIABLES")
print("=" * 80)

categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in ['loan_status', 'default']]

for col in categorical_cols:
    if df[col].dtype == 'category':
        df[col] = df[col].astype(str)

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=float)
print(f"\nShape after encoding: {df_encoded.shape}")

# Ensure all columns are numeric
remaining_cat = df_encoded.select_dtypes(include=['object', 'category']).columns.tolist()
if remaining_cat:
    for col in remaining_cat:
        df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce')
        df_encoded[col].fillna(0, inplace=True)

print("[OK] All columns are numeric")


PHASE 5: ENCODING CATEGORICAL VARIABLES

Shape after encoding: (10000, 145)
[OK] All columns are numeric


#### What It Means

This converts categorical variables (like state, home_ownership, loan_purpose) into numeric format using one-hot encoding. After encoding, the dataset goes from 57 columns to 145 columns.

#### Explain the Decision
- **One-hot encoding** Creates binary columns for each category. Required for most ML models.
- **Drop first category** Avoids multicollinearity (the "dummy variable trap").
- **145 columns after encoding** Shows the dataset has expanded significantly — many categorical features.

**I understand how to prepare categorical data for machine learning models.**

####  PHASE 6: PREPARE FEATURES AND TARGET

### Why this phase exists

The features (`X`) and target (`y`) are separated here, and the feature names are
cleaned so they are compatible with the ML libraries.

**What to notice in the output:** `X` has 143 columns. That is a large number
relative to 10,000 rows, and most of those columns are the one-hot encoded state
dummies. A model with more features than signal can overfit; the cross-validation
results in Phase 10 will show whether that happened here.

**The feature-name cleaning** is a small but real governance point. Some ML
libraries silently mangle or reject column names containing characters like `[`,
`<`, or spaces. Cleaning them explicitly — rather than letting the library do it
unpredictably — keeps the feature set stable and traceable across reruns.

In [6]:
print("\n" + "=" * 80)
print("PHASE 6: PREPARING FEATURES AND TARGET")
print("=" * 80)

# Separate features and target
X = df_encoded.drop(columns=['loan_status', 'default'])
y = df_encoded['default']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Default rate: {y.mean():.2%}")

# Clean feature names
def clean_feature_names(df):
    """Clean feature names for compatibility with ML libraries."""
    df_clean = df.copy()
    import re
    for col in df_clean.columns:
        clean_col = str(col)
        clean_col = clean_col.replace('[', '_').replace(']', '_')
        clean_col = clean_col.replace('<', '_').replace('>', '_')
        clean_col = clean_col.replace(' ', '_').replace('-', '_')
        clean_col = clean_col.replace('(', '_').replace(')', '_')
        clean_col = clean_col.replace('/', '_').replace('?', '_')
        clean_col = clean_col.replace('!', '_').replace('@', '_')
        clean_col = clean_col.replace('#', '_').replace('$', '_')
        clean_col = clean_col.replace('%', '_').replace('^', '_')
        clean_col = clean_col.replace('&', '_').replace('*', '_')
        clean_col = re.sub(r'_+', '_', clean_col)
        clean_col = clean_col.rstrip('_')
        if clean_col and clean_col[0].isdigit():
            clean_col = 'f_' + clean_col
        if clean_col != col:
            df_clean.rename(columns={col: clean_col}, inplace=True)
    return df_clean

X = clean_feature_names(X)
print("[OK] Feature names cleaned")


PHASE 6: PREPARING FEATURES AND TARGET
X shape: (10000, 143)
y shape: (10000,)
Default rate: 1.78%
[OK] Feature names cleaned


### PHASE 7: SPLIT DATA AND CREATE TRAIN/VALIDATION SETS

### Why this phase exists

The data is split into a training set (80%) and a validation set (20%). The
validation set is held back and used to evaluate performance on data the model
has never seen.

**The critical detail — and the honest gap:** For credit risk models, the
*correct* split is temporal. You train on loans originated before a cutoff date
and validate on loans originated after it. That mirrors how the model will
actually be used: it is trained on the past and applied to the future.

This notebook uses a **random split instead**, because no usable date column
survives the cleaning in Notebook 01. The output says so explicitly: *"No date
column found - using random split (fallback)."*

**Why this matters, stated plainly:** A random split lets the model see loans from
the same time period in both training and validation. If economic conditions were
similar across that period, the validation overstates how well the model will
perform when conditions change. A temporal split is a stronger test. Its absence
is not a fatal flaw, but it is a limitation — and it is documented as such in
Phase 16 and Finding 3.

**The governance lesson:** A fallback is not the same as the intended design.
What makes this defensible is that the fallback is *labelled* as a fallback, not
presented as the intended approach.

In [7]:
print("\n" + "=" * 80)
print("PHASE 7: SPLITTING DATA")
print("=" * 80)

# Split data
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")

# Check for date column (for time-based split documentation)
date_col_used = None  # No date column available
print("[INFO] No date column found - using random split (fallback)")


PHASE 7: SPLITTING DATA
X_train shape: (8000, 143)
X_val shape: (2000, 143)
y_train shape: (8000,)
y_val shape: (2000,)
[INFO] No date column found - using random split (fallback)


### PHASE 8: FORCE VALIDATE NO NaN VALUES (NOW X_train AND X_val EXIST)

### Why this phase exists

This phase is defensive: it asserts, explicitly, that no missing values remain in
the training or validation data before any model sees it.

**Why this is worth its own phase:** Missing values are the most common cause of
a model training run failing silently or producing nonsense. Different libraries
handle NaN differently — some impute, some drop, some error, some propagate. By
forcing all NaN to be resolved *before* training, the notebook removes that
ambiguity.

**What the output reveals:** `emp_years` is entirely NaN in both the training and
validation sets, so it is dropped. That confirms the note from Phase 4 — the
feature engineering step produced a column that did not survive. The phase catches
it, drops it, and moves on. The `assert` statements at the end are the enforcement
mechanism: if any NaN remained, the notebook would stop rather than continue with
bad data.

In [8]:
print("\n" + "=" * 80)
print("PHASE 8: FORCE VALIDATE NO NaN VALUES (CRITICAL)")
print("=" * 80)

def validate_no_nan(data, name):
    """Force-validate that a dataset has no NaN values."""
    if data is None:
        print(f"[ERROR] {name} is None!")
        return False
    
    if hasattr(data, 'isnull'):
        total_nan = data.isnull().sum().sum()
        if total_nan > 0:
            print(f"[ERROR] {name} contains {total_nan} NaN values!")
            nan_cols = data.columns[data.isnull().any()].tolist()
            print(f"  Columns with NaN: {nan_cols[:10]}")
            for col in nan_cols[:5]:
                nan_count = data[col].isnull().sum()
                print(f"    {col}: {nan_count} NaN values")
            return False
        else:
            print(f"[OK] {name} has no NaN values (shape: {data.shape})")
            return True
    return False

def fix_all_nan_columns(data, name):
    """Remove all-NaN columns and return cleaned data."""
    all_nan_cols = data.columns[data.isnull().all()].tolist()
    if all_nan_cols:
        print(f"[INFO] {name} has {len(all_nan_cols)} all-NaN columns: {all_nan_cols}")
        data = data.drop(columns=all_nan_cols)
        print(f"  {name} shape after dropping all-NaN columns: {data.shape}")
    return data

def safe_impute_data(data, name):
    """Safely impute data without shape mismatch."""
    # First, remove all-NaN columns
    data = fix_all_nan_columns(data, name)
    
    # Check if any NaN remain
    if data.isnull().sum().sum() > 0:
        print(f"[INFO] Imputing remaining NaN values in {name}...")
        from sklearn.impute import SimpleImputer
        imputer = SimpleImputer(strategy='median')
        data_imputed = imputer.fit_transform(data)
        
        # Create DataFrame with correct columns
        data = pd.DataFrame(data_imputed, columns=data.columns, index=data.index)
        print(f"  {name} shape after imputation: {data.shape}")
        print(f"  {name} NaN after imputation: {data.isnull().sum().sum()}")
    else:
        print(f"[OK] {name} has no NaN values")
    
    return data

# Validate and fix X_train
print("\n" + "=" * 40)
print("FIXING X_TRAIN:")
print("=" * 40)
X_train = safe_impute_data(X_train, "X_train")
validate_no_nan(X_train, "X_train (after fix)")

# Validate and fix X_val
print("\n" + "=" * 40)
print("FIXING X_VAL:")
print("=" * 40)
X_val = safe_impute_data(X_val, "X_val")
validate_no_nan(X_val, "X_val (after fix)")

# Validate y_train and y_val
print("\n" + "=" * 40)
print("VALIDATING Y_TRAIN AND Y_VAL:")
print("=" * 40)

if y_train.isnull().sum() > 0:
    print(f"[ERROR] y_train contains {y_train.isnull().sum()} NaN values!")
    y_train = y_train.dropna()
    print(f"  y_train shape after dropping NaN: {y_train.shape}")
else:
    print("[OK] y_train has no NaN values")

if y_val.isnull().sum() > 0:
    print(f"[ERROR] y_val contains {y_val.isnull().sum()} NaN values!")
    y_val = y_val.dropna()
    print(f"  y_val shape after dropping NaN: {y_val.shape}")
else:
    print("[OK] y_val has no NaN values")

# Final validation
print("\n" + "=" * 40)
print("FINAL VALIDATION SUMMARY:")
print("=" * 40)
print(f"X_train shape: {X_train.shape}, NaN: {X_train.isnull().sum().sum()}")
print(f"X_val shape: {X_val.shape}, NaN: {X_val.isnull().sum().sum()}")
print(f"y_train shape: {y_train.shape}, NaN: {y_train.isnull().sum()}")
print(f"y_val shape: {y_val.shape}, NaN: {y_val.isnull().sum()}")

# Assert no missing values
assert X_train.isnull().sum().sum() == 0, "X_train still contains NaN values!"
assert X_val.isnull().sum().sum() == 0, "X_val still contains NaN values!"
assert y_train.isnull().sum() == 0, "y_train still contains NaN values!"
assert y_val.isnull().sum() == 0, "y_val still contains NaN values!"

print("\n[OK] All data validated — NO NaN values present")


PHASE 8: FORCE VALIDATE NO NaN VALUES (CRITICAL)

FIXING X_TRAIN:
[INFO] X_train has 1 all-NaN columns: ['emp_years']
  X_train shape after dropping all-NaN columns: (8000, 142)
[OK] X_train has no NaN values
[OK] X_train (after fix) has no NaN values (shape: (8000, 142))

FIXING X_VAL:
[INFO] X_val has 1 all-NaN columns: ['emp_years']
  X_val shape after dropping all-NaN columns: (2000, 142)
[OK] X_val has no NaN values
[OK] X_val (after fix) has no NaN values (shape: (2000, 142))

VALIDATING Y_TRAIN AND Y_VAL:
[OK] y_train has no NaN values
[OK] y_val has no NaN values

FINAL VALIDATION SUMMARY:
X_train shape: (8000, 142), NaN: 0
X_val shape: (2000, 142), NaN: 0
y_train shape: (8000,), NaN: 0
y_val shape: (2000,), NaN: 0

[OK] All data validated — NO NaN values present


### PHASE 9: HANDLE CLASS IMBALANCE

### Why this phase exists

The target is severely imbalanced: only 1.78% of loans default. A model trained
without correction will learn to predict "no default" almost always, achieving
high accuracy and catching almost no defaults. That is useless for a collections
early-warning system.

**How the imbalance is corrected:** by weighting the classes so the minority class
carries more influence during training. The output shows the weights — class 1
gets a weight roughly 55× class 0. That is a strong correction, appropriate for a
strong imbalance.

**The governance consequence:** class weighting is a modeling decision that
affects the trade-off between false positives and false negatives. A model tuned
to catch more defaults will also flag more good loans as risky. That trade-off is
not a technical detail — it is a business and fairness decision, and the low
precision reported in Phase 12 (0.03–0.04) is a direct consequence of it.

**A note on the `scale_pos_weight` for XGBoost:** it is set to 55.34, matching
the class ratio. Different libraries handle imbalance in slightly different ways,
and the notebook documents the choice for each. A reviewer should be able to see
that the imbalance handling was deliberate and consistent, not incidental.

In [9]:
print("\n" + "=" * 80)
print("PHASE 9: HANDLING CLASS IMBALANCE")
print("=" * 80)

from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

print(f"\nClass Weights:")
print(f"  Class 0: {class_weight_dict[0]:.2f}")
print(f"  Class 1: {class_weight_dict[1]:.2f}")
print(f"  XGBoost scale_pos_weight: {scale_pos_weight:.2f}")


PHASE 9: HANDLING CLASS IMBALANCE

Class Weights:
  Class 0: 0.51
  Class 1: 28.17
  XGBoost scale_pos_weight: 55.34


#### What It Means

The dataset has a 1.78% default rate (highly imbalanced). This section calculates class weights to help the model pay attention to the minority class (defaults).

#### Explain the Decision
- **Balanced class weights** The model sees far more "good" loans than "bad" loans. Class weights increase the penalty for misclassifying defaults.
- **Class 0 weight:** 0.51 The majority class (performing loans) gets a lower weight.
- **Class 1 weight:** 28.17 The minority class (defaults) gets a much higher weight.
- **XGBoost scale_pos_weight:** 55.34	The ratio of good to bad loans — used by XGBoost to handle imbalance.

**I understand class imbalance and how to handle it. I don't just ignore it.**

### PHASE 10: MODEL TRAINING WITH 5-FOLD CROSS-VALIDATION

### Why this phase exists

Four candidate models are trained — Logistic Regression, XGBoost, Random Forest,
and LightGBM — and each is evaluated with 5-fold cross-validation.

**Why four models rather than one:** The point of comparing models is not
necessarily to find a winner; it is to understand how sensitive the results are
to the modeling choice. If four reasonable models all produce AUC around 0.6–0.67,
that is a more honest picture of the achievable performance than any single
number. If one model were dramatically better, that would be worth investigating
— and might indicate overfitting or a methodological issue.

**How to read the cross-validation output:** The CV AUC means are 0.669 (Logistic
Regression), 0.657 (Random Forest), 0.638 (LightGBM), and 0.609 (XGBoost). Notice
that Logistic Regression — the simplest model — has the *highest* cross-validated
AUC. That is a meaningful result: with 143 features and 8,000 training rows, the
more complex models may be overfitting.

**A quiet inconsistency worth flagging:** the cross-validation ranking (Logistic
Regression first) does not match the validation-set ranking (XGBoost first). This
is not necessarily a problem — cross-validation and holdout evaluation answer
slightly different questions — but it is the kind of thing a reviewer will ask
about, and the notebook does not currently address it.

In [10]:
# ============================================================================
# 10.1 INITIALIZE MODELS DICTIONARY
# ============================================================================

models = {}
predictions = {}
cv_results = {}

In [11]:
# ============================================================================
# 10.2 LOGISTIC REGRESSION
# ============================================================================

print("\nTraining Logistic Regression...")
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', LogisticRegression(
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE,
        max_iter=1000,
        C=1.0,
        solver='lbfgs'
    ))
])

cv_scores_lr = cross_val_score(
    logistic_pipeline, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc'
)
print(f"CV AUC-ROC: {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std()*2:.4f})")

logistic_pipeline.fit(X_train, y_train)
models['Logistic Regression'] = logistic_pipeline
predictions['Logistic Regression'] = logistic_pipeline.predict_proba(X_val)[:, 1]
cv_results['Logistic Regression'] = cv_scores_lr
print("[OK] Logistic Regression trained")


Training Logistic Regression...
CV AUC-ROC: 0.6693 (+/- 0.0568)
[OK] Logistic Regression trained


In [12]:
# ============================================================================
# 10.3 XGBOOST
# ============================================================================

print("\nTraining XGBoost...")
# XGBoost handles NaN natively, but ensure no NaN for safety
X_train_clean = X_train.fillna(0)
X_val_clean = X_val.fillna(0)

import xgboost as xgb

model_xgb = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    use_label_encoder=False,
    verbosity=0
)

cv_scores_xgb = cross_val_score(
    model_xgb, X_train_clean, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc'
)
print(f"CV AUC-ROC: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std()*2:.4f})")

model_xgb.fit(X_train_clean, y_train)
models['XGBoost'] = model_xgb
predictions['XGBoost'] = model_xgb.predict_proba(X_val_clean)[:, 1]
cv_results['XGBoost'] = cv_scores_xgb
print("[OK] XGBoost trained")


Training XGBoost...
CV AUC-ROC: 0.6093 (+/- 0.0718)
[OK] XGBoost trained


In [13]:
# ============================================================================
# 10.4 RANDOM FOREST
# ============================================================================

print("\nTraining Random Forest...")
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    class_weight=class_weight_dict,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

cv_scores_rf = cross_val_score(
    model_rf, X_train_clean, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc'
)
print(f"CV AUC-ROC: {cv_scores_rf.mean():.4f} (+/- {cv_scores_rf.std()*2:.4f})")

model_rf.fit(X_train_clean, y_train)
models['Random Forest'] = model_rf
predictions['Random Forest'] = model_rf.predict_proba(X_val_clean)[:, 1]
cv_results['Random Forest'] = cv_scores_rf
print("[OK] Random Forest trained")


Training Random Forest...
CV AUC-ROC: 0.6569 (+/- 0.0582)
[OK] Random Forest trained


In [14]:
# ============================================================================
# 10.5 LIGHTGBM
# ============================================================================

print("\nTraining LightGBM...")
import lightgbm as lgb

model_lgb = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=4,
    class_weight=class_weight_dict,
    random_state=RANDOM_STATE,
    verbose=-1,
    n_jobs=1
)

cv_scores_lgb = cross_val_score(
    model_lgb, X_train_clean, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc'
)
print(f"CV AUC-ROC: {cv_scores_lgb.mean():.4f} (+/- {cv_scores_lgb.std()*2:.4f})")

model_lgb.fit(X_train_clean, y_train)
models['LightGBM'] = model_lgb
predictions['LightGBM'] = model_lgb.predict_proba(X_val_clean)[:, 1]
cv_results['LightGBM'] = cv_scores_lgb
print("[OK] LightGBM trained")


Training LightGBM...
CV AUC-ROC: 0.6375 (+/- 0.0598)
[OK] LightGBM trained


In [15]:
# ============================================================================
# 10.6 CROSS-VALIDATION SUMMARY
# ============================================================================

print("\n" + "=" * 40)
print("CROSS-VALIDATION SUMMARY:")
print("=" * 40)

cv_df = pd.DataFrame({
    'Model': cv_results.keys(),
    'CV_AUC_Mean': [np.mean(scores) for scores in cv_results.values()],
    'CV_AUC_Std': [np.std(scores) for scores in cv_results.values()]
})
print(cv_df.round(4).to_string(index=False))

print("\n[OK] All models trained successfully")



CROSS-VALIDATION SUMMARY:
              Model  CV_AUC_Mean  CV_AUC_Std
Logistic Regression       0.6693      0.0284
            XGBoost       0.6093      0.0359
      Random Forest       0.6569      0.0291
           LightGBM       0.6375      0.0299

[OK] All models trained successfully


#### What It Means

**This trains 4 models (Logistic Regression, XGBoost, Random Forest, LightGBM) using 5-fold cross-validation. The CV AUC-ROC scores show how well each model generalizes.**

#### Explain the Decision
- **4 models trained** Comparing multiple models ensures you find the best performer, not just the first one that works.
- **5-fold cross-validation** Tests the model on 5 different splits of the data, reducing the chance of overfitting.
- **CV AUC-ROC scores** Shows generalization performance — not just training performance.
- **Logistic Regression CV:** 0.669	Baseline model — interpretable but lower performance.
- **XGBoost CV:** 0.609	More powerful but lower CV score here (interesting).
- **Random Forest CV:** 0.657	Good ensemble method.
- **LightGBM CV:** 0.638	Efficient gradient boosting.

### PHASE 11: LEAKAGE IMPACT QUANTIFICATION (BEFORE/AFTER COMPARISON)

### Why this phase exists

This is the single most important phase in the notebook, and it is not a modeling
phase at all. It is the phase where the model's original performance claim is
tested and found to be misleading.

**The finding:** The developer's original model, which included the nine leakage
features, reported AUC = 0.865. The validated model, with those features removed,
reports a substantially lower AUC. The phase quantifies the gap and states the
conclusion plainly: the original performance claim is *invalid*.

**Why the gap exists:** The leakage features contain information that only exists
*after* a loan is issued — repayment amounts, months since last delinquency,
account balances. A model that uses them is not predicting default; it is reading
the outcome. The 0.865 figure was not a measure of predictive skill; it was a
measure of how well the model could see the answer.

**The honest number is the one that survives removal.** That number is weaker,
and that is the point. A model that reports 0.62 honestly is more valuable than
one that reports 0.87 dishonestly, because only the first can be trusted in
production.

**Where the notebook should be more careful — and where a reviewer will look:**
The leakage drop is reported as **22.6%** in the output but as **28.2%** in the
docstrings, the executive summary, and the findings register. And the "realistic
AUC" is computed as **0.669** in this phase (from the best cross-validation mean)
but reported as **0.621** in the Phase 12 performance table (from the best
validation-set AUC). Those two numbers answer slightly different questions, but
presenting both without explaining the difference invites confusion.

**What would strengthen this phase:** A single sentence reconciling the two
figures, and a decision about which one is the official "realistic baseline." A
GRC reviewer will not hold the discrepancy against the project if it is
acknowledged; they will hold it against the project if it is not.

In [16]:
print("\n" + "=" * 80)
print("PHASE 11: LEAKAGE IMPACT QUANTIFICATION (CRITICAL)")
print("=" * 80)

"""
===============================================================================
LEAKAGE IMPACT QUANTIFICATION
===============================================================================

INDEPENDENT VALIDATION FINDING #1: DATA LEAKAGE IMPACT

Finding: 9 leakage features were identified and removed from the model.

Impact Quantification:
- With leakage features:     AUC-ROC = 0.865 (INFLATED)
- Without leakage features:  AUC-ROC = 0.621 (REALISTIC)
- Performance drop:          0.244 AUC points (28.2% inflation)

DEFINITION OF "REALISTIC AUC"
=============================
Throughout this notebook, "Realistic AUC" is defined as:

    The highest AUC-ROC achieved by any model on the HELD-OUT
    VALIDATION SET, after removal of leakage features.

Rationale:
1. Held-out validation is the closest analogue to production use:
   the model is trained on history and applied to unseen applicants.
2. It is the number a validator can independently reproduce.
3. It is the most direct answer to "how well will this model perform
   on new data?"

Supporting figure (not the headline):
   The best cross-validated AUC on the TRAINING set was 0.669
   (Logistic Regression). This estimates ranking ability within the
   training distribution and is reported for reference only. It is
   NOT the realistic performance figure, because it is not measured
   on held-out data.

Why the two numbers differ:
   0.669 (best CV mean) estimates within-training-distribution ranking.
   0.621 (best held-out AUC) measures performance on unseen data.
   The latter is the more appropriate headline for validation.

REGULATORY REFERENCE:
- OSFI E-23 Section 3.3: Data Quality (leakage is a data quality issue)
- SR 11-7: Model validation must identify data quality issues

===============================================================================
"""

# ----------------------------------------------------------------------------
# SINGLE SOURCE OF TRUTH: REALISTIC_AUC
# ----------------------------------------------------------------------------
# This variable is defined ONCE here and referenced throughout the notebook.
# Do not redefine it elsewhere. If the model set or evaluation changes,
# update it here and every downstream reference stays consistent.

LEAKAGE_AUC = 0.865  # Developer's original model WITH leakage features (invalid)

# Compute the realistic AUC: best held-out validation AUC across all models.
# Falls back to 0.621 if metrics_df is not yet available (it is computed in
# Phase 12, so at this point in the notebook we derive it from `predictions`).
try:
    _val_aucs = {
        name: roc_auc_score(y_val, preds)
        for name, preds in predictions.items()
    }
    REALISTIC_AUC = max(_val_aucs.values())
    REALISTIC_AUC_MODEL = max(_val_aucs, key=_val_aucs.get)
except (NameError, ValueError):
    # Defensive fallback if predictions/y_val are unavailable
    REALISTIC_AUC = 0.621
    REALISTIC_AUC_MODEL = "XGBoost"
    _val_aucs = {}

LEAKAGE_DELTA = LEAKAGE_AUC - REALISTIC_AUC
LEAKAGE_PERCENT = (LEAKAGE_DELTA / LEAKAGE_AUC) * 100

print(f"""
===============================================================================
LEAKAGE IMPACT ANALYSIS
===============================================================================

Model with Leakage Features (Developer's Original):
    AUC-ROC: {LEAKAGE_AUC:.3f}
    Status:  INVALID — not usable for new applications

Model without Leakage Features (Validated Model):
    AUC-ROC: {REALISTIC_AUC:.3f}
    Model:   {REALISTIC_AUC_MODEL}
    Basis:   Best held-out validation AUC across all candidate models

Performance Difference:
    Absolute:   {LEAKAGE_DELTA:.3f} AUC points
    Percentage: {LEAKAGE_PERCENT:.1f}% inflation

===============================================================================
DEFINITION — "REALISTIC AUC"
===============================================================================

Realistic AUC = best AUC-ROC on the HELD-OUT VALIDATION SET,
                after removal of all leakage features.

    Realistic AUC = {REALISTIC_AUC:.3f}  ({REALISTIC_AUC_MODEL})

For reference, the best CROSS-VALIDATED AUC on the training set
was {max(np.mean(s) for s in cv_results.values()):.3f} ({max(cv_results, key=lambda k: np.mean(cv_results[k]))}).
That figure is NOT the realistic performance number — it estimates
ranking ability within the training distribution, not performance
on unseen data.

===============================================================================
VALIDATION OPINION
===============================================================================

The developer's original performance claim (AUC {LEAKAGE_AUC:.3f}) is INVALID.
The true performance of a leakage-free model is AUC {REALISTIC_AUC:.3f}.

This is a CRITICAL finding that must be reported to the Model Risk Committee.

Recommendation: The model may be used for new applications ONLY with
the leakage-free version. The inflated performance must not be used
in any business reporting.
""")

# Save leakage impact analysis
leakage_impact_df = pd.DataFrame({
    'Metric': [
        'AUC with Leakage',
        'AUC without Leakage (Realistic AUC)',
        'Absolute Drop',
        'Percentage Drop'
    ],
    'Value': [LEAKAGE_AUC, REALISTIC_AUC, LEAKAGE_DELTA, f"{LEAKAGE_PERCENT:.1f}%"],
    'Interpretation': [
        'INFLATED (not valid for new applications)',
        f'REALISTIC (valid for new applications) — {REALISTIC_AUC_MODEL}',
        f'Cost of data leakage: {LEAKAGE_DELTA:.3f} AUC',
        f'Performance inflated by {LEAKAGE_PERCENT:.1f}%'
    ]
})
leakage_impact_df.to_csv("outputs/reports/leakage_impact_analysis.csv", index=False)
print("[OK] Leakage impact analysis saved to outputs/reports/leakage_impact_analysis.csv")


PHASE 11: LEAKAGE IMPACT QUANTIFICATION (CRITICAL)

LEAKAGE IMPACT ANALYSIS

Model with Leakage Features (Developer's Original):
    AUC-ROC: 0.865
    Status:  INVALID — not usable for new applications

Model without Leakage Features (Validated Model):
    AUC-ROC: 0.621
    Model:   XGBoost
    Basis:   Best held-out validation AUC across all candidate models

Performance Difference:
    Absolute:   0.244 AUC points
    Percentage: 28.2% inflation

DEFINITION — "REALISTIC AUC"

Realistic AUC = best AUC-ROC on the HELD-OUT VALIDATION SET,
                after removal of all leakage features.

    Realistic AUC = 0.621  (XGBoost)

For reference, the best CROSS-VALIDATED AUC on the training set
was 0.669 (Logistic Regression).
That figure is NOT the realistic performance number — it estimates
ranking ability within the training distribution, not performance
on unseen data.

VALIDATION OPINION

The developer's original performance claim (AUC 0.865) is INVALID.
The true performance of a

### PHASE 12: PERFORMANCE EVALUATION & REALISTIC DOCUMENTATION

### Why this phase exists

This phase reports the honest performance of each model: AUC-ROC, PR-AUC,
Brier score, log loss, accuracy, precision, recall, F1, and the optimal decision
threshold.

**Why so many metrics:** No single number captures whether a credit model is
useful. AUC measures ranking ability; PR-AUC measures performance on the
minority (default) class; Brier and log loss measure probability quality;
precision and recall capture the business trade-off. A model can look good on one
and bad on another, and the only way to see that is to report several.

**What the output shows, and what it means:**

- **AUC-ROC around 0.59–0.62.** This is the honest, leakage-free performance. It
  is modest — a meaningful but far-from-decisive signal.
- **PR-AUC around 0.03–0.06.** With a 1.78% base rate, random guessing would give
  a PR-AUC near 0.018. The models are above that, but not dramatically.
- **Precision around 0.03–0.04.** This is the uncomfortable number. It means that
  of the loans the model flags as likely to default, only about 3–4% actually do.
  The rest are false positives.
- **Recall around 0.36–0.78.** Different models trade off differently; LightGBM
  catches the most defaults (recall 0.78) at the cost of the lowest accuracy.

**Why the low precision is not necessarily a failure:** For a collections
early-warning system, a false positive means a customer gets a courtesy call or
a reminder — a low-cost intervention. A false negative means a default is missed.
The business may rationally prefer high recall and low precision. But that
trade-off has to be a *business decision*, documented and owned, not a technical
accident.

**The governance point:** The notebook reports precision honestly rather than
hiding it behind accuracy. That is the right instinct. The next step — which
belongs in the governance report, not the notebook — is to state explicitly what
level of precision the business accepts, and why.

In [17]:
print("\n" + "=" * 80)
print("PHASE 12: PERFORMANCE EVALUATION (REALISTIC)")
print("=" * 80)

def calculate_metrics(y_true, y_proba, name):
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    youden_j = tpr - fpr
    optimal_idx = np.argmax(youden_j)
    optimal_threshold = thresholds[optimal_idx] if len(thresholds) > 0 else 0.5
    
    y_pred = (y_proba >= optimal_threshold).astype(int)
    
    metrics = {
        'Model': name,
        'AUC-ROC': roc_auc_score(y_true, y_proba),
        'PR-AUC': average_precision_score(y_true, y_proba),
        'Brier': brier_score_loss(y_true, y_proba),
        'Log_Loss': log_loss(y_true, y_proba),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        'Optimal_Threshold': optimal_threshold
    }
    return metrics

results = []
for name in models.keys():
    metrics = calculate_metrics(y_val, predictions[name], name)
    results.append(metrics)

metrics_df = pd.DataFrame(results)
print("\n" + "=" * 80)
print("MODEL COMPARISON (REALISTIC PERFORMANCE)")
print("=" * 80)
print(metrics_df.round(4).to_string(index=False))

# V2.0: Document performance drop (references REALISTIC_AUC from Phase 11)
print("\n" + "=" * 80)
print("PERFORMANCE NOTE (V2.0)")
print("=" * 80)

best_model = metrics_df.loc[metrics_df['AUC-ROC'].idxmax(), 'Model']
best_auc = metrics_df['AUC-ROC'].max()

assert abs(best_auc - REALISTIC_AUC) < 1e-6 or best_auc == REALISTIC_AUC, (
    f"Inconsistency: best validation AUC ({best_auc:.4f}) does not match "
    f"REALISTIC_AUC ({REALISTIC_AUC:.4f}) defined in Phase 11."
)

print(f"""
PERFORMANCE COMPARISON:
- With leakage features (inflated): AUC-ROC {LEAKAGE_AUC:.3f}
- Without leakage features (realistic): AUC-ROC {REALISTIC_AUC:.3f}
- Best model by held-out AUC: {best_model}
- Performance drop: {LEAKAGE_DELTA:.3f} AUC points ({LEAKAGE_PERCENT:.1f}% inflation)
- This is the COST OF DATA LEAKAGE

DEFINITION REMINDER:
- "Realistic AUC" = best held-out validation AUC after leakage removal.
- The best cross-validated AUC on training data was
  {max(np.mean(s) for s in cv_results.values()):.3f} ({max(cv_results, key=lambda k: np.mean(cv_results[k]))}),
  reported for reference only.

INTERPRETATION:
- The model is NOT as strong as originally reported.
- The realistic AUC-ROC of {REALISTIC_AUC:.3f} reflects genuine predictive power.
- The model CAN be used for new applications (no leakage).
- Performance should be tracked against this baseline.

RECOMMENDATION:
- Do NOT compare to inflated historical metrics.
- Establish monitoring baseline at {REALISTIC_AUC:.3f} AUC.
- Consider model enhancements to improve performance.
""")

cv_df = pd.DataFrame({
    'Model': cv_results.keys(),
    'CV_AUC_Mean': [np.mean(scores) for scores in cv_results.values()],
    'CV_AUC_Std': [np.std(scores) for scores in cv_results.values()]
})
print("\nCROSS-VALIDATION RESULTS (5-FOLD):")
print(cv_df.round(4).to_string(index=False))


PHASE 12: PERFORMANCE EVALUATION (REALISTIC)

MODEL COMPARISON (REALISTIC PERFORMANCE)
              Model  AUC-ROC  PR-AUC  Brier  Log_Loss  Accuracy  Precision  Recall  F1-Score  Optimal_Threshold
Logistic Regression   0.5887  0.0635 0.2112    0.6142    0.7950     0.0348  0.3889    0.0639             0.5557
            XGBoost   0.6210  0.0619 0.0715    0.2555    0.7155     0.0316  0.5000    0.0595             0.2316
      Random Forest   0.6106  0.0435 0.1280    0.4319    0.8395     0.0418  0.3611    0.0749             0.4545
           LightGBM   0.6130  0.0336 0.0848    0.2920    0.4265     0.0240  0.7778    0.0466             0.1223

PERFORMANCE NOTE (V2.0)

PERFORMANCE COMPARISON:
- With leakage features (inflated): AUC-ROC 0.865
- Without leakage features (realistic): AUC-ROC 0.621
- Best model by held-out AUC: XGBoost
- Performance drop: 0.244 AUC points (28.2% inflation)
- This is the COST OF DATA LEAKAGE

DEFINITION REMINDER:
- "Realistic AUC" = best held-out validation AUC

#### What It Means

This evaluates the leakage-free models on the validation set. The best model is XGBoost with AUC = 0.621. The performance is MODERATE (not excellent, but usable for early warning).

#### Explain the Decision
- **Realistic AUC:** 0.621	This is the true performance of the model. It's what you'd get in production.
- **PR-AUC:** 0.062	Precision-Recall AUC is low because the default rate is very low. This is expected.
- **Recall:** 0.500	The model catches 50% of defaults. This is the trade-off for low precision.
- **Precision:** 0.032	Only 3.2% of predicted defaults are actual defaults. This means many false positives.
- **Best Model:** XGBoost	XGBoost has the highest AUC (0.621) among the 4 models.

Establish monitoring baseline at 0.621	This is the baseline for future performance monitoring.

**I document realistic performance, not inflated performance. I set a baseline for monitoring.**

### PHASE 13: CALIBRATION ASSESSMENT

### Why this phase exists

A model can rank applicants correctly (good AUC) and still produce probabilities
that are systematically wrong. Calibration is the property that a predicted
probability of 0.10 actually corresponds to roughly a 10% observed default rate.

**Why calibration matters more than it might seem:** If the model's probabilities
are used for anything quantitative — pricing, provisioning, capital allocation —
they must be calibrated. A well-ranked but poorly calibrated model can produce
probabilities that are off by a factor of two or more.

**How calibration is assessed here:** three measures — calibration slope (target:
1.0), calibration intercept (target: 0.0), and the Hosmer-Lemeshow test (target:
p > 0.05).

**What the output shows:**

- **Logistic Regression:** HL p-value 1.0000 — good calibration.
- **Random Forest:** HL p-value 1.0000 — good calibration.
- **LightGBM:** HL p-value 0.2972 — acceptable calibration.
- **XGBoost:** HL p-value 0.0000 — *poor* calibration.

**The uncomfortable result:** XGBoost is the "best" model by validation AUC
(0.621) but is the worst-calibrated. That is a genuine tension, and the notebook
should not paper over it. A model that ranks well but produces untrustworthy
probabilities is not automatically the best choice.

**What the output also shows:** all four models have calibration intercepts around
-4.3 to -4.6, far from the target of 0.0. That is a large systematic
under-prediction of default probability across the board — a consequence of the
severe class imbalance and the strong class weighting. It is the kind of thing
that would need correction (Platt scaling or isotonic regression) before the model
could be used quantitatively.

**One thing to note about the HL test:** the p-values of exactly 1.0000 for two
models are suspicious — they usually indicate the test statistic is near zero,
which can happen when the binning is degenerate. Worth a second look; the
notebook presents the number without interrogating it.

In [18]:
print("\n" + "=" * 80)
print("PHASE 13: CALIBRATION ASSESSMENT")
print("=" * 80)

def assess_calibration(y_true, y_proba, name, n_bins=10):
    from scipy.stats import chi2
    
    fraction_positive, mean_predicted = calibration_curve(
        y_true, y_proba, n_bins=n_bins, strategy='quantile'
    )
    
    bins = np.percentile(y_proba, np.linspace(0, 100, n_bins+1))
    bin_indices = np.digitize(y_proba, bins[1:-1])
    
    observed = []
    expected = []
    for i in range(n_bins):
        mask = bin_indices == i
        if mask.sum() > 0:
            observed.append(y_true[mask].sum())
            expected.append(y_proba[mask].sum())
    
    hl_stat = 0
    for obs, exp in zip(observed, expected):
        if exp > 0 and exp < len(observed):
            hl_stat += ((obs - exp) ** 2) / (exp * (1 - exp/len(observed)))
    
    df = n_bins - 2
    p_value = 1 - chi2.cdf(hl_stat, df) if df > 0 else 1.0
    
    from sklearn.linear_model import LogisticRegression
    cal_model = LogisticRegression()
    cal_model.fit(y_proba.reshape(-1, 1), y_true)
    cal_slope = cal_model.coef_[0][0]
    cal_intercept = cal_model.intercept_[0]
    
    return {
        'Model': name,
        'Calibration_Slope': cal_slope,
        'Calibration_Intercept': cal_intercept,
        'HL_Statistic': hl_stat,
        'HL_p_value': p_value,
        'Mean_Predicted': mean_predicted,
        'Fraction_Positive': fraction_positive
    }

calibration_results = []
for name in models.keys():
    cal_result = assess_calibration(y_val, predictions[name], name)
    calibration_results.append(cal_result)

print("\nCALIBRATION METRICS:")
print("Perfect calibration: Slope=1, Intercept=0, HL p-value > 0.05")
print("=" * 60)

for result in calibration_results:
    print(f"\n{result['Model']}:")
    print(f"  Calibration Slope: {result['Calibration_Slope']:.4f} (target: 1.0)")
    print(f"  Calibration Intercept: {result['Calibration_Intercept']:.4f} (target: 0.0)")
    print(f"  Hosmer-Lemeshow p-value: {result['HL_p_value']:.4f} (target: > 0.05)")
    
    if result['HL_p_value'] > 0.05:
        print("  [OK] Good calibration (p-value > 0.05)")
    else:
        print("  [WARNING] Poor calibration (p-value <= 0.05)")


PHASE 13: CALIBRATION ASSESSMENT

CALIBRATION METRICS:
Perfect calibration: Slope=1, Intercept=0, HL p-value > 0.05

Logistic Regression:
  Calibration Slope: 1.2935 (target: 1.0)
  Calibration Intercept: -4.5852 (target: 0.0)
  Hosmer-Lemeshow p-value: 1.0000 (target: > 0.05)
  [OK] Good calibration (p-value > 0.05)

XGBoost:
  Calibration Slope: 1.3779 (target: 1.0)
  Calibration Intercept: -4.2771 (target: 0.0)
  Hosmer-Lemeshow p-value: 0.0000 (target: > 0.05)
  [WARNING] Poor calibration (p-value <= 0.05)

Random Forest:
  Calibration Slope: 1.2108 (target: 1.0)
  Calibration Intercept: -4.4151 (target: 0.0)
  Hosmer-Lemeshow p-value: 1.0000 (target: > 0.05)
  [OK] Good calibration (p-value > 0.05)

LightGBM:
  Calibration Slope: 1.1453 (target: 1.0)
  Calibration Intercept: -4.2626 (target: 0.0)
  Hosmer-Lemeshow p-value: 0.2972 (target: > 0.05)
  [OK] Good calibration (p-value > 0.05)


#### What It Means

This checks whether the model's predicted probabilities are well-calibrated (i.e., a 10% predicted probability means 10% actual default rate).

#### Explain the Decision
- **Hosmer-Lemeshow test** Tests whether predicted probabilities match actual outcomes.
- **XGBoost HL p-value = 0.0000** XGBoost is poorly calibrated — probabilities are not reliable.
- **Calibration Slope:** 1.38	Slope > 1 means the model is underconfident — predicted probabilities are too low.
- **Calibration Intercept:** -4.28	Negative intercept means the model is systematically underestimating risk.
- **Logistic Regression HL = 1.0000** Logistic Regression is well-calibrated (p-value > 0.05).
- Random Forest HL = 1.0000** Random Forest is also well-calibrated.
- LightGBM HL = 0.2972** LightGBM is well-calibrated (p-value > 0.05).

**A high AUC doesn't mean good calibration. I test both discrimination (AUC) and calibration (HL test). XGBoost has the best AUC but worst calibration.**

### PHASE 14: FAIRNESS / BIAS TESTING

### Why this phase exists

A model can be accurate on average and still treat groups differently in ways
that are legally or ethically unacceptable. This phase tests for that.

**What is being tested, and how:** The notebook uses **state** as a geographic
proxy for protected class, because Lending Club does not provide race or gender
data. For each state, it computes the approval rate and the disparate impact (DI)
ratio — the ratio of the lowest group's approval rate to the highest group's.

**The convention:** DI > 0.8 is generally considered acceptable (the "four-fifths
rule" from U.S. fair lending law). DI between 0.6 and 0.8 is a warning zone.
DI below 0.6 is severe.

**What the output shows:** severe disparate impact in multiple states.

- `state_AL`: DI = 0.528
- `state_AR`: DI = 0.555
- `state_AZ`: DI = 0.244

All three are well below 0.6. The minimum DI is 0.244 — meaning the least-approved
group is approved at roughly one-quarter the rate of the most-approved group.

**What this means, stated plainly:** On this test, the model is not fair with
respect to geography. Deployment is blocked until this is addressed.

**The remediation options, as the notebook lists them:** remove state features,
apply reweighing (the technique the remediation article describes), or apply
fairness constraints during training. The notebook does not implement any of them
— that is the work of Notebook 03 and beyond.

**Two honest caveats the notebook states, and that a reader should hold onto:**

1. **State is an imperfect proxy for race/ethnicity.** The DI numbers are
   *indicative*, not definitive. They tell you there is a geographic disparity;
   they do not tell you it is a racial disparity. That distinction matters
   legally and analytically.
2. **The analysis is limited to three states** (`state_AL`, `state_AR`, `state_AZ`)
   out of many. The code limits the loop to the first three protected columns for
   performance. That is a reasonable engineering choice, but it means the fairness
   picture is incomplete. The notebook reports what it tested; it should be clear
   that it did not test everything.

**The governance significance:** This finding — DI < 0.6 — is what turns Notebook
02 from "here is a model" into "here is a model that cannot be deployed yet, and
here is why." That is a far more valuable portfolio artifact than a clean bill of
health would be. It shows the governance process catching something real.

In [19]:
print("\n" + "=" * 80)
print("PHASE 14: FAIRNESS TESTING WITH PROTECTED CLASS PROXY (V2.0)")
print("=" * 80)

"""
===============================================================================
FAIRNESS TESTING — PROTECTED CLASS PROXY DOCUMENTATION (V2.0)
===============================================================================

DECISION POINT 5: PROTECTED CLASS PROXY
========================================
Decision: Use state as a geographic proxy for protected class analysis

Rationale:
1. No explicit race/gender data available in Lending Club dataset
2. Geographic location (state) is a recognized proxy for demographic diversity
3. State-level analysis is a common approach in fair lending testing

Protected Class Proxied:
- Race/Ethnicity (geographic proxy)
- Geographic diversity

Limitation:
- State is an imperfect proxy for race/ethnicity
- Results should be interpreted as indicative, not definitive
- Additional testing recommended with explicit protected attributes

Regulatory Reference:
- ECOA: Fair lending requires testing for disparate impact
- CFPB: Geographic proxies are acceptable for fair lending testing
- OSFI E-23 Section 5.2: Fairness/Bias testing

===============================================================================
"""

print("\n12.1 PROTECTED CLASS PROXY DOCUMENTATION:")
print("-" * 40)

print("Protected Class Proxy Documentation:")
print("  - Proxy Variable: State (geographic location)")
print("  - Protected Class Proxied: Race/Ethnicity, Geographic Diversity")
print("  - Threshold: 4/5ths Rule (80%)")
print("  - Severity Classification: DI < 0.6 = SEVERE, 0.6-0.8 = MODERATE, > 0.8 = NONE")
print()
print("Regulatory Reference: ECOA, CFPB, OSFI E-23 Section 5.2")

def comprehensive_fairness_analysis(X_data, y_true, y_pred, model):
    fairness_report = []
    
    protected_cols = []
    for col in X_data.columns:
        if any(term in col.lower() for term in ['state', 'gender', 'sex', 'race', 'ethnicity']):
            if X_data[col].nunique() <= 20:
                protected_cols.append(col)
    
    if not protected_cols:
        np.random.seed(RANDOM_STATE)
        X_data['Gender_Synthetic'] = np.random.choice([0, 1], size=len(X_data), p=[0.6, 0.4])
        protected_cols = ['Gender_Synthetic']
        print("  [NOTE] No protected attributes found. Using synthetic gender for demonstration.")
    
    print(f"\n  Found protected attributes: {protected_cols[:5]}... (showing first 5)")
    
    for attr in protected_cols[:3]:  # Limit to first 3 for performance
        groups = X_data[attr].unique()
        group_metrics = []
        
        for group in groups[:5]:
            mask = X_data[attr] == group
            if mask.sum() > 10:
                y_true_group = y_true[mask]
                y_pred_group = y_pred[mask]
                
                tp = ((y_true_group == 1) & (y_pred_group == 1)).sum()
                fp = ((y_true_group == 0) & (y_pred_group == 1)).sum()
                fn = ((y_true_group == 1) & (y_pred_group == 0)).sum()
                tn = ((y_true_group == 0) & (y_pred_group == 0)).sum()
                
                tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
                fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                
                group_metrics.append({
                    'Attribute': attr,
                    'Group': str(group)[:20],
                    'N': mask.sum(),
                    'TPR': tpr,
                    'FPR': fpr,
                    'Precision': precision,
                    'Approval_Rate': y_pred_group.mean(),
                    'Default_Rate': y_true_group.mean()
                })
        
        if len(group_metrics) >= 2:
            g_df = pd.DataFrame(group_metrics)
            print(f"\n  Fairness Analysis by {attr}:")
            print(g_df.round(4).to_string(index=False))
            
            min_approval = g_df['Approval_Rate'].min()
            max_approval = g_df['Approval_Rate'].max()
            di_ratio = min_approval / max_approval if max_approval > 0 else 0
            
            print(f"\n  Disparate Impact Ratio: {di_ratio:.4f}")
            if di_ratio > 0.8:
                print("    [OK] No disparate impact (>= 0.8)")
            elif di_ratio > 0.6:
                print("    [WARNING] Monitor disparate impact (0.6-0.8)")
            else:
                print("    [ERROR] Severe disparate impact (< 0.6) - mitigation required")
            
            fairness_report.append({
                'Attribute': attr,
                'DI_Ratio': di_ratio,
                'Group_Count': len(g_df)
            })
    
    return pd.DataFrame(fairness_report) if fairness_report else pd.DataFrame()

print("\n12.2 FAIRNESS ASSESSMENT:")
print("-" * 50)

best_model_instance = models[best_model]
y_pred_best = best_model_instance.predict(X_val)

fairness_df = comprehensive_fairness_analysis(X_val, y_val, y_pred_best, best_model_instance)

# V2.0: Enhanced fairness documentation
print("\n" + "=" * 80)
print("FAIRNESS FINDINGS (V2.0)")
print("=" * 80)

if not fairness_df.empty:
    min_di = fairness_df['DI_Ratio'].min()
    print(f"\nMINIMUM DISPARATE IMPACT: {min_di:.4f}")
    
    if min_di < 0.6:
        print("\n[CRITICAL] SEVERE DISPARATE IMPACT DETECTED")
        print("  - DI < 0.6 in multiple states")
        print("  - Model CANNOT be deployed without fairness mitigation")
        print("\nREQUIRED ACTIONS:")
        print("  1. Remove state features OR")
        print("  2. Apply reweighting (Kamiran & Calders) OR")
        print("  3. Apply fairness constraints during training")
        print("  4. Document mitigation in governance report")
    
    print("\nFAIRNESS SUMMARY:")
    print(fairness_df.round(4).to_string(index=False))


PHASE 14: FAIRNESS TESTING WITH PROTECTED CLASS PROXY (V2.0)

12.1 PROTECTED CLASS PROXY DOCUMENTATION:
----------------------------------------
Protected Class Proxy Documentation:
  - Proxy Variable: State (geographic location)
  - Protected Class Proxied: Race/Ethnicity, Geographic Diversity
  - Threshold: 4/5ths Rule (80%)
  - Severity Classification: DI < 0.6 = SEVERE, 0.6-0.8 = MODERATE, > 0.8 = NONE

Regulatory Reference: ECOA, CFPB, OSFI E-23 Section 5.2

12.2 FAIRNESS ASSESSMENT:
--------------------------------------------------

  Found protected attributes: ['state_AL', 'state_AR', 'state_AZ', 'state_CA', 'state_CO']... (showing first 5)

  Fairness Analysis by state_AL:
Attribute Group    N    TPR    FPR  Precision  Approval_Rate  Default_Rate
 state_AL   0.0 1964 0.2222 0.0705     0.0556         0.0733        0.0183
 state_AL   1.0   36 0.0000 0.1389     0.0000         0.1389        0.0000

  Disparate Impact Ratio: 0.5279
    [ERROR] Severe disparate impact (< 0.6) - mi

#### What It Means

This tests whether the model has disparate impact across protected classes. Since Lending Club doesn't have race/gender data, state is used as a geographic proxy for race/ethnicity.

#### Explain the Decision
- **State as proxy for race/ethnicity** Geographic location is a recognized proxy for demographic diversity. This is an acceptable approach under ECOA/CFPB guidance.
- **4/5ths rule (80% threshold)** The ECOA standard — if one group's approval rate is <80% of another group's rate, it's evidence of adverse impact.
- **DI: 0.2435 (SEVERE)** The minimum Disparate Impact ratio is 0.2435 — far below 0.8. This is SEVERE disparate impact.
- **Multiple states show severe DI	state_AL:** 0.5279, state_AR: 0.5554, state_AZ: 0.2435 — all below 0.6.
- **DEPLOYMENT BLOCKED** This is a CRITICAL finding — the model cannot be deployed without fairness mitigation.

**I understand fair lending requirements. I test for disparate impact. When I find it, I don't ignore it — I flag it as a deployment blocker.**

### PHASE 15: SAVE MODELS AND RESULTS

### Why this phase exists

Everything the notebook produced — trained models, predictions, metrics,
calibration results, a performance summary — is saved to disk so that Notebook 03
can load and independently validate it.

**Why this matters for governance:** The validation notebook must be able to
reproduce and check the development notebook's results *without rerunning it*.
If the artifacts were not saved, the validator would have to trust the
development notebook's output rather than verify it. Saving artifacts is what
makes independent validation possible.

**What to notice:** the performance summary JSON includes the leakage drop, the
realistic AUC, the best model, and the fairness status. It is designed to be read
by a downstream process — or a reviewer — without needing to open the notebook.

In [20]:
print("\n" + "=" * 80)
print("PHASE 15: SAVING MODELS AND RESULTS")
print("=" * 80)

for name, model in models.items():
    filename = f"models/{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(model, filename)
    print(f"[OK] Saved: {filename}")

test_data = {
    'X_val': X_val,
    'y_val': y_val,
    'features': X_train.columns.tolist(),
    'date_col_used': date_col_used,
    'split_type': 'time-based' if date_col_used else 'random (fallback)'
}
joblib.dump(test_data, "models/test_data.pkl")
print("[OK] Saved: models/test_data.pkl")

joblib.dump(predictions, "models/predictions.pkl")
print("[OK] Saved: models/predictions.pkl")

metrics_df.to_csv("models/performance_metrics.csv", index=False)
print("[OK] Saved: models/performance_metrics.csv")

cv_df.to_csv("models/cv_results.csv", index=False)
print("[OK] Saved: models/cv_results.csv")

calibration_save_df = pd.DataFrame([{
    'Model': r['Model'],
    'Calibration_Slope': r['Calibration_Slope'],
    'Calibration_Intercept': r['Calibration_Intercept'],
    'HL_Statistic': r['HL_Statistic'],
    'HL_p_value': r['HL_p_value']
} for r in calibration_results])
calibration_save_df.to_csv("models/calibration_metrics.csv", index=False)
print("[OK] Saved: models/calibration_metrics.csv")

# V2.0: Save performance summary
performance_summary = {
    'realistic_auc': REALISTIC_AUC,
    'realistic_auc_definition': 'Best held-out validation AUC after leakage removal',
    'realistic_auc_model': REALISTIC_AUC_MODEL,
    'inflated_auc': LEAKAGE_AUC,
    'performance_drop': LEAKAGE_DELTA,
    'performance_drop_percent': LEAKAGE_PERCENT,
    'leakage_features_removed': leakage_features_removed,
    'best_model': best_model,
    'min_disparate_impact': float(fairness_df['DI_Ratio'].min()) if not fairness_df.empty else None,
    'fairness_status': 'CRITICAL' if (not fairness_df.empty and fairness_df['DI_Ratio'].min() < 0.6) else 'MONITOR' if (not fairness_df.empty and fairness_df['DI_Ratio'].min() < 0.8) else 'OK',
    'split_type': 'time-based' if date_col_used else 'random (fallback)'
}

with open("outputs/reports/performance_summary_v2.json", "w") as f:
    json.dump(performance_summary, f, indent=2)
print("[OK] Saved: outputs/reports/performance_summary_v2.json")


PHASE 15: SAVING MODELS AND RESULTS
[OK] Saved: models/logistic_regression.pkl
[OK] Saved: models/xgboost.pkl
[OK] Saved: models/random_forest.pkl
[OK] Saved: models/lightgbm.pkl
[OK] Saved: models/test_data.pkl
[OK] Saved: models/predictions.pkl
[OK] Saved: models/performance_metrics.csv
[OK] Saved: models/cv_results.csv
[OK] Saved: models/calibration_metrics.csv
[OK] Saved: outputs/reports/performance_summary_v2.json


#### What It Means

This saves all trained models, performance metrics, and validation data so the next notebook (Independent Validation) can use them.

##### Explain the Decision
- **Save models as .pkl files** Allows independent validation in Notebook 3. Creates an audit trail.
- **Save test_data.pkl** Contains the validation set so the independent validator uses the same data.
- **Save performance_summary_v2.json** Documents key metrics in a portable format.

**I save my work so it can be independently validated. I create an audit trail.**

### PHASE 16: NOTEBOOK LIMITATIONS (V2.0)

### Why this phase exists

This phase lists, in one place, every limitation of the notebook: the realistic
performance ceiling, the fairness blocker, the missing time-based split, the IFRS 9
misalignment, the foreign-data gap, the calibration issues, the proxy weakness,
and the low precision.

**Why a limitations section is not an admission of failure:** A model development
notebook that claims no limitations is either not looking for them or not
reporting them. Both are worse than having limitations. The governance value of
this phase is that a reader can see exactly what the model does not do, without
having to infer it.

**The connection to the remediation article:** This section is the *identify*
stage of the five-stage feedback loop. Notebook 03 will *validate*; Notebook 04
will *document*; the monitoring stage lives outside the notebooks. What this
phase does is make sure nothing is forgotten between stages.

**One thing a reviewer will notice:** the limitations section contains f-string
placeholders that were not formatted correctly in the output — for example,
`{fairness_df['DI_Ratio'].min():.4f}` appears literally rather than as a number.
That is a bug, not a governance issue, but it is visible in the output and worth
fixing before this notebook is used as a portfolio piece.

In [21]:
print("\n" + "=" * 80)
print("PHASE 16: NOTEBOOK LIMITATIONS (V2.0)")
print("=" * 80)

_min_di = fairness_df['DI_Ratio'].min() if not fairness_df.empty else float('nan')
_split_note = "Time-based split implemented" if date_col_used else "Random split used as fallback"
_split_detail = f"Using date column: {date_col_used}" if date_col_used else "No date column found in data"
_split_impact = "Valid out-of-time validation" if date_col_used else "Potential look-ahead bias"
_split_mitigation = "Maintain time-based split for all future runs" if date_col_used else "Add date column for future versions"

print(f"""
===============================================================================
NOTEBOOK 02 LIMITATIONS (V2.0)
===============================================================================

1. PERFORMANCE — REALISTIC AUC-ROC IS {REALISTIC_AUC:.3f} (NOT {LEAKAGE_AUC:.3f})
   --------------------------------------------------------------
   After removing 9 leakage features, the best held-out validation
   AUC-ROC is {REALISTIC_AUC:.3f} ({REALISTIC_AUC_MODEL}).
   This is the COST OF DATA LEAKAGE — a {LEAKAGE_PERCENT:.1f}% inflation.

   DEFINITION: "Realistic AUC" = best held-out validation AUC after
   leakage removal. It is NOT the best cross-validated training AUC
   ({max(np.mean(s) for s in cv_results.values()):.3f}), which measures
   ranking within the training distribution.

   IMPACT: The model is weaker than originally reported.

   MITIGATION: Document realistic performance baseline. Do not use
   inflated metrics in business reporting.

2. FAIRNESS — SEVERE DISPARATE IMPACT DETECTED
   --------------------------------------------
   Minimum Disparate Impact ratio is {_min_di:.4f} (DI < 0.6).
   Model CANNOT be deployed without fairness mitigation.

   IMPACT: Deployment blocked until fairness is addressed.

   MITIGATION: Remove state features or apply reweighting.

3. TIME-BASED SPLIT
   -----------------
   {_split_note}
   {_split_detail}

   IMPACT: {_split_impact}

   MITIGATION: {_split_mitigation}

4. IFRS 9 MISALIGNMENT — 16+ DAY THRESHOLD
   ----------------------------------------
   Model uses 16+ days late as default definition. IFRS 9 requires 30+ days.
   NOT suitable for provisioning.

   IMPACT: Cannot be used for IFRS 9 provisioning.

   MITIGATION: Recalibrate for 30+ day threshold if needed.

5. FOREIGN DATA — U.S. DATA ONLY
   ------------------------------
   Developed on U.S. Lending Club data. Not validated for Canadian portfolios.

   IMPACT: Cannot deploy in Canada without validation.

   MITIGATION: Validate on Canadian data before deployment.

6. CALIBRATION — SOME MODELS POORLY CALIBRATED
   --------------------------------------------
   Multiple models show HL p-value <= 0.05, indicating poor calibration.

   IMPACT: Predicted probabilities are not reliable.

   MITIGATION: Apply Platt scaling or isotonic regression.

7. PROTECTED ATTRIBUTES — NO EXPLICIT RACE/GENDER DATA
   ---------------------------------------------------
   State used as geographic proxy. This is an imperfect proxy.

   IMPACT: Fairness testing results are indicative, not definitive.

   MITIGATION: Add explicit protected attributes if available.

8. PRECISION — LOW PRECISION (0.03-0.04)
   --------------------------------------
   Precision is very low, indicating high false positive rate.

   IMPACT: Many good loans would be flagged as defaults.

   MITIGATION: Review threshold (0.5) for business acceptance.
   Consider business cost of false positives vs false negatives.

===============================================================================
""")


PHASE 16: NOTEBOOK LIMITATIONS (V2.0)

NOTEBOOK 02 LIMITATIONS (V2.0)

1. PERFORMANCE — REALISTIC AUC-ROC IS 0.621 (NOT 0.865)
   --------------------------------------------------------------
   After removing 9 leakage features, the best held-out validation
   AUC-ROC is 0.621 (XGBoost).
   This is the COST OF DATA LEAKAGE — a 28.2% inflation.

   DEFINITION: "Realistic AUC" = best held-out validation AUC after
   leakage removal. It is NOT the best cross-validated training AUC
   (0.669), which measures
   ranking within the training distribution.

   IMPACT: The model is weaker than originally reported.

   MITIGATION: Document realistic performance baseline. Do not use
   inflated metrics in business reporting.

2. FAIRNESS — SEVERE DISPARATE IMPACT DETECTED
   --------------------------------------------
   Minimum Disparate Impact ratio is 0.2435 (DI < 0.6).
   Model CANNOT be deployed without fairness mitigation.

   IMPACT: Deployment blocked until fairness is addressed.

   

#### What It Means

This documents the limitations of the analysis — what assumptions were made and what the impact is.

#### Explain the Decision
- **Document limitations** Shows intellectual honesty — you're not claiming perfection.
- **Include impact and mitigation** For each limitation, you explain why it matters and what you did about it.
- **Foreign data limitation** Acknowledges this is U.S. data, not Canadian.
- **Time-based split gap** Acknowledges that random split was used (fallback) — needs improvement.

**I am intellectually honest. I acknowledge limitations and explain how they mitigate them."

### PHASE 17: EXECUTIVE SUMMARY

### Why this phase exists

The executive summary is what a busy reviewer reads if they read nothing else. It
states the best model, the realistic AUC, the inflated AUC, the performance drop,
the leakage features removed, the split type, the fairness status, and the key
findings.

**Why it is placed at the end rather than the beginning:** The summary is a
*synthesis*, not an introduction. It can only be written after the work is done.
Its position at the end is honest about that.

**What the summary does well:** It leads with the uncomfortable numbers — the
performance drop, the fairness blocker — rather than burying them. A summary that
leads with "model developed successfully" and mentions the fairness problem in
paragraph seven is a summary that is managing the reader rather than informing
them.

**What the summary should reconcile:** As noted in Phase 11, the summary reports
a 28.2% inflation while the leakage-impact output reports 22.6%, and a realistic
AUC of 0.621 while the leakage-impact phase computes 0.669. These are the kinds of
inconsistencies that, once noticed, cast doubt on everything else. Fixing them —
or explaining them — would materially strengthen the artifact.

In [22]:
print("\n" + "=" * 80)
print("PHASE 17: EXECUTIVE SUMMARY (V2.0)")
print("=" * 80)

print(f"""
================================================================================
MODEL DEVELOPMENT COMPLETE (V2.0)
================================================================================

BEST MODEL: {best_model}
REALISTIC AUC-ROC: {REALISTIC_AUC:.4f}  (best held-out validation AUC, {REALISTIC_AUC_MODEL})
INFLATED AUC-ROC: {LEAKAGE_AUC:.3f}
PERFORMANCE DROP: {LEAKAGE_DELTA:.3f} ({LEAKAGE_PERCENT:.1f}% inflation)

LEAKAGE FEATURES REMOVED: {len(leakage_features_removed)}
TIME-BASED SPLIT: {"✅ Implemented" if date_col_used else "⚠️ Fallback (random)"}
FAIRNESS STATUS: {"CRITICAL" if (not fairness_df.empty and fairness_df['DI_Ratio'].min() < 0.6) else "MONITOR" if (not fairness_df.empty and fairness_df['DI_Ratio'].min() < 0.8) else "OK"}

MODEL PERFORMANCE:
==================
{metrics_df.round(4).to_string(index=False)}

CROSS-VALIDATION (5-FOLD):
==========================
{cv_df.round(4).to_string(index=False)}

KEY FINDINGS (V2.0):
====================
1. [CRITICAL] Leakage removal caused {LEAKAGE_PERCENT:.1f}% performance drop ({LEAKAGE_AUC:.3f} → {REALISTIC_AUC:.3f})
2. [CRITICAL] Severe fairness concerns (DI < 0.6) block deployment
3. [HIGH] Time-based split {"implemented" if date_col_used else "needs implementation"}
4. [HIGH] IFRS 9 misalignment (16+ vs 30+ days)
5. [MEDIUM] Calibration issues in some models
6. [MEDIUM] Low precision (0.03-0.04)

FILES GENERATED (V2.0):
=======================
1. models/*.pkl - Trained model files
2. models/performance_metrics.csv - Performance metrics
3. models/cv_results.csv - Cross-validation results
4. models/calibration_metrics.csv - Calibration metrics
5. models/test_data.pkl - Validation data for Notebook 03
6. outputs/reports/performance_summary_v2.json - Summary
7. outputs/reports/leakage_impact_analysis.csv - Before/after comparison
8. outputs/figures/*.png - All visualizations

NEXT STEPS:
===========
1. Proceed to Notebook 03 (Independent Validation)
2. Validate realistic performance (AUC 0.621)
3. Document fairness mitigation requirements
4. Prepare validation report with findings
5. Report to Model Risk Committee

================================================================================
MODEL DEVELOPMENT V2.0 COMPLETE
================================================================================
""")

print("\n[OK] Model Development V2.0 Complete")



PHASE 17: EXECUTIVE SUMMARY (V2.0)

MODEL DEVELOPMENT COMPLETE (V2.0)

BEST MODEL: XGBoost
REALISTIC AUC-ROC: 0.6210  (best held-out validation AUC, XGBoost)
INFLATED AUC-ROC: 0.865
PERFORMANCE DROP: 0.244 (28.2% inflation)

LEAKAGE FEATURES REMOVED: 9
TIME-BASED SPLIT: ⚠️ Fallback (random)
FAIRNESS STATUS: CRITICAL

MODEL PERFORMANCE:
              Model  AUC-ROC  PR-AUC  Brier  Log_Loss  Accuracy  Precision  Recall  F1-Score  Optimal_Threshold
Logistic Regression   0.5887  0.0635 0.2112    0.6142    0.7950     0.0348  0.3889    0.0639             0.5557
            XGBoost   0.6210  0.0619 0.0715    0.2555    0.7155     0.0316  0.5000    0.0595             0.2316
      Random Forest   0.6106  0.0435 0.1280    0.4319    0.8395     0.0418  0.3611    0.0749             0.4545
           LightGBM   0.6130  0.0336 0.0848    0.2920    0.4265     0.0240  0.7778    0.0466             0.1223

CROSS-VALIDATION (5-FOLD):
              Model  CV_AUC_Mean  CV_AUC_Std
Logistic Regression       0.

### What It Means

This summarizes everything accomplished in the notebook — key metrics, findings, and next steps.

#### Explain the Decision
- **Best Model:** XGBoost	Clear recommendation for the best model.
- **Realistic AUC:** 0.621	The true performance of the model.
- **Performance drop: 22.6%	The cost of removing leakage.
- **Severe fairness concerns** Deployment blocker — must be addressed.
- **Next steps** Shows you're thinking ahead — the project doesn't stop here.

**I produce deliverables — not just code. I summarize key findings for stakeholder.**

### PHASE 18: FINDINGS REGISTER (V2.0)

### Why this phase exists

The findings register is the governance consolidation: every issue identified in
this notebook, with severity, evidence reference, remediation, owner, timeline,
and status.

**What is in it:** six findings. One resolved (the leakage removal, with a
complete remediation). Five open — the fairness blocker, the IFRS 9 misalignment,
the foreign-data gap, the calibration issue, and the low precision.

**What the register demonstrates, and why that matters for a GRC portfolio:**
The register is the mechanism by which technical findings become managed items.
A finding without an owner is a complaint; a finding with an owner, a timeline,
and a status is a governed risk. This notebook has five open governed risks. That
is a realistic picture of a model in development.

**The connection to Notebook 01:** Notebook 01's register contained seven
findings, three of which were resolved within that notebook. Notebook 02 inherits
the open ones, adds new ones (fairness, calibration, precision), and resolves one
of the critical ones (leakage). Notebook 03 will inherit this register and add
its own validation findings. The register is cumulative and traceable across the
notebook series — which is exactly what an audit trail is supposed to look like.

**What a reviewer should take from this:** Not "the model has problems," but "the
project has a process for finding, tracking, and resolving problems." That is the
distinction that matters.

In [23]:
print("\n" + "=" * 80)
print("PHASE 18: FINDINGS REGISTER (V2.0)")
print("=" * 80)

findings = [
    {
        "id": 1,
        "severity": "CRITICAL",
        "finding": "Data Leakage — 9 features removed, 28.2% performance inflation",
        "description": f"Features including balance, paid_total, months_since_last_delinq, and issue_month were not available at origination. Removal caused AUC drop from {LEAKAGE_AUC:.3f} to {REALISTIC_AUC:.3f} ({LEAKAGE_PERCENT:.1f}% inflation). Realistic AUC is defined as the best held-out validation AUC after leakage removal.",
        "evidence_reference": "Notebook 01, Section 6 — Leakage Detection; Notebook 02, Section 9 — Leakage Impact Quantification",
        "remediation": "Document realistic performance baseline. Do not use inflated metrics.",
        "owner": "Model Development Team",
        "status": "RESOLVED",
        "timeline": "Complete (V2.0)"
    },
    {
        "id": 2,
        "severity": "CRITICAL",
        "finding": "Severe Fairness Concerns — Disparate Impact < 0.6",
        "description": "State-level disparate impact detected with DI ratio < 0.6 in multiple states.",
        "evidence_reference": "Notebook 02, Section 12 — Fairness Testing",
        "remediation": "Remove state features OR apply reweighting OR apply fairness constraints.",
        "owner": "Model Risk Team",
        "status": "OPEN",
        "timeline": "3 months"
    },
    {
        "id": 3,
        "severity": "HIGH",
        "finding": "IFRS 9 Staging Misalignment — 16+ Day Threshold",
        "description": "Model uses 16+ days late as default definition. IFRS 9 requires 30+ days.",
        "evidence_reference": "Notebook 01, Section 4 — Target Definition; Notebook 02, Section 0 — Intended Use",
        "remediation": "Recalibrate for 30+ day threshold if used for IFRS 9.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "3 months"
    },
    {
        "id": 4,
        "severity": "HIGH",
        "finding": "Foreign Data Applicability — U.S. Data Only",
        "description": "Model developed on U.S. Lending Club data. Not validated for Canadian portfolios.",
        "evidence_reference": "Notebook 01, Section 2 — Dataset Caveat",
        "remediation": "Validate on Canadian data before deployment.",
        "owner": "Model Risk Team",
        "status": "OPEN",
        "timeline": "6 months"
    },
    {
        "id": 5,
        "severity": "MEDIUM",
        "finding": "Poor Calibration — HL p-value ≤ 0.05 for multiple models",
        "description": "Several models show poor calibration with Hosmer-Lemeshow p-value ≤ 0.05.",
        "evidence_reference": "Notebook 02, Section 11 — Calibration Assessment",
        "remediation": "Apply Platt scaling or isotonic regression.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "1 month"
    },
    {
        "id": 6,
        "severity": "MEDIUM",
        "finding": "Low Precision — High False Positive Rate",
        "description": "Precision is 0.03-0.04, indicating many false positives.",
        "evidence_reference": "Notebook 02, Section 10 — Performance Evaluation",
        "remediation": "Review threshold and business acceptance criteria.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "1 month"
    }
]

# Display findings
print("\nFINDINGS REGISTER (V2.0):")
print("=" * 80)

for f in findings:
    status_icon = "✅" if f['status'] == 'RESOLVED' else "❌"
    print(f"\nFinding {f['id']}: {f['finding']}")
    print(f"  Severity: {f['severity']} {status_icon}")
    print(f"  Evidence: {f['evidence_reference']}")
    print(f"  Remediation: {f['remediation']}")
    print(f"  Owner: {f['owner']}")
    print(f"  Timeline: {f['timeline']}")
    print(f"  Status: {f['status']}")

# Save findings register
findings_df = pd.DataFrame(findings)
findings_df.to_csv("outputs/reports/findings_register_v2.csv", index=False)
print("\n[OK] Findings register saved to outputs/reports/findings_register_v2.csv")

print("\n[OK] Model Development V2.0 Complete")


PHASE 18: FINDINGS REGISTER (V2.0)

FINDINGS REGISTER (V2.0):

Finding 1: Data Leakage — 9 features removed, 28.2% performance inflation
  Severity: CRITICAL ✅
  Evidence: Notebook 01, Section 6 — Leakage Detection; Notebook 02, Section 9 — Leakage Impact Quantification
  Remediation: Document realistic performance baseline. Do not use inflated metrics.
  Owner: Model Development Team
  Timeline: Complete (V2.0)
  Status: RESOLVED

Finding 2: Severe Fairness Concerns — Disparate Impact < 0.6
  Severity: CRITICAL ❌
  Evidence: Notebook 02, Section 12 — Fairness Testing
  Remediation: Remove state features OR apply reweighting OR apply fairness constraints.
  Owner: Model Risk Team
  Timeline: 3 months
  Status: OPEN

Finding 3: IFRS 9 Staging Misalignment — 16+ Day Threshold
  Severity: HIGH ❌
  Evidence: Notebook 01, Section 4 — Target Definition; Notebook 02, Section 0 — Intended Use
  Remediation: Recalibrate for 30+ day threshold if used for IFRS 9.
  Owner: Model Development Team


#### What It Means

This documents 6 findings from the analysis, each with:

- Severity (Critical/High/Medium)
- Evidence (where in the notebooks this was found)
- Remediation (what needs to be done)
- Owner (who is responsible)
- Timeline (when it should be done)

#### Explain the Decision
- **Findings register** Creates an audit trail. Regulators want to see findings documented, not just discovered.
- **Severity rating** Helps prioritize what needs to be fixed first.
- **Evidence reference** Shows traceability — you can point to exactly where the finding was discovered.
- **Owner assigned** Shows accountability — someone is responsible for fixing it.
- **2 resolved, 4 open** Shows progress — some issues have been fixed, others remain.

**I don't just find issues — I document them with severity, evidence, remediation, owner, and timeline. This is exactly what a regulator expects.**

## What this notebook establishes, and what it hands off

This notebook has done five things:

1. **Removed nine leakage features** and quantified the cost — a lower, honest
   performance figure instead of an inflated one.
2. **Trained and compared four models**, reporting performance honestly across
   multiple metrics rather than selecting the most flattering number.
3. **Assessed calibration** and found that the best-performing model by AUC is
   also the worst-calibrated — a tension worth carrying forward.
4. **Tested for fairness** and found severe geographic disparate impact, blocking
   deployment until mitigated.
5. **Consolidated six findings** into a register with owners and timelines.

What it has *not* done is fix anything. The fairness blocker is documented, not
resolved. The calibration issue is noted, not corrected. The time-based split is
desired, not implemented.

**That is the correct division of labor.** Notebook 02's job is to build and
characterize the model honestly. Notebook 03's job is to independently validate
it. Notebook 04's job is to document the whole thing for governance. Notebook 05
tests whether the remediation approach generalizes.

The thread connecting all five notebooks is the same one that runs through this
one: **the honest number is more valuable than the flattering one, and a finding
that is documented is a finding that can be managed.**